# SVM и градиентный бустинг: проверяем руками

Ноутбук к теме [`lesson.md`](lesson.md). Здесь SVM собирается с нуля — сначала
через двойственную задачу, потом через hinge loss и субградиентный спуск — и
сверяется с sklearn; ядерный трюк проверяется численно; опорные векторы
отслеживаются при изменении $C$ и $\gamma$; отступы AdaBoost считаются вручную;
и отдельно проверяется, что бустинг не умеет экстраполировать.

In [1]:
import time
import warnings

import numpy as np
from scipy.optimize import minimize
from sklearn.datasets import make_circles, make_classification, make_moons
from sklearn.ensemble import AdaBoostClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

W_TRUE = np.array([1.0, 0.6]) / np.linalg.norm([1.0, 0.6])

def separable(n, rng, gap=1.2, lim=3.0):
    '''Два класса по разные стороны прямой W_TRUE·x = 0 с зазором gap.'''
    X, y = [], []
    while len(X) < n:
        p = rng.uniform(-lim, lim, 2)
        s = p @ W_TRUE
        if abs(s) > gap / 2:
            X.append(p); y.append(1 if s > 0 else -1)
    return np.array(X), np.array(y)

## 1. Жёсткий SVM через двойственную задачу

$$\max_\alpha\ \sum_i\alpha_i - \tfrac12\sum_{i,j}\alpha_i\alpha_j y_i y_j\langle x_i, x_j\rangle
\quad\text{при}\quad \alpha_i \ge 0,\ \sum_i \alpha_i y_i = 0.$$

Решаем обычным оптимизатором и восстанавливаем $w = \sum_i \alpha_i y_i x_i$, а $b$ —
по опорным векторам. Затем сравниваем со `SVC(kernel="linear", C=1e6)`.

In [2]:
rng = np.random.default_rng(0)
X, y = separable(30, rng)
n = len(y)
Q = (y[:, None] * y[None, :]) * (X @ X.T)

res = minimize(
    fun=lambda a: 0.5 * a @ Q @ a - a.sum(),
    x0=np.zeros(n),
    jac=lambda a: Q @ a - 1,
    constraints=[{"type": "eq", "fun": lambda a: a @ y, "jac": lambda a: y.astype(float)}],
    bounds=[(0, None)] * n,
    method="SLSQP",
    options={"maxiter": 500, "ftol": 1e-12},
)
alpha = res.x
sv = alpha > 1e-6
w = (alpha * y) @ X
b = np.mean(y[sv] - X[sv] @ w)

sk = SVC(kernel="linear", C=1e6).fit(X, y)
w_sk, b_sk = sk.coef_[0], sk.intercept_[0]

print(f"оптимизатор сошёлся: {res.success}")
print(f"опорные векторы (свой):    {np.sort(np.where(sv)[0])}")
print(f"опорные векторы (sklearn): {np.sort(sk.support_)}")
print(f"w свой    = {np.round(w, 4)},  b = {b:.4f}")
print(f"w sklearn = {np.round(w_sk, 4)},  b = {b_sk:.4f}")
print(f"ширина полосы: {2 / np.linalg.norm(w):.4f} против {2 / np.linalg.norm(w_sk):.4f}")
print(f"min y(w·x + b) = {(y * (X @ w + b)).min():.4f}  (на опорных векторах должно быть 1)")
print(f"ненулевых α: {sv.sum()} из {n} — решение зависит только от них")

оптимизатор сошёлся: True
опорные векторы (свой):    [10 14 15]
опорные векторы (sklearn): [10 14 15]
w свой    = [1.0333 0.6118],  b = 0.2394
w sklearn = [1.0333 0.6118],  b = 0.2394
ширина полосы: 1.6655 против 1.6655
min y(w·x + b) = 1.0000  (на опорных векторах должно быть 1)
ненулевых α: 3 из 30 — решение зависит только от них


## 2. Мягкий SVM как hinge loss + $L_2$

sklearn минимизирует $\tfrac12\|w\|^2 + C\sum_i\max(0, 1 - y_i f(x_i))$. Та же задача
субградиентным спуском — ни квадратичного программирования, ни двойственности.

In [3]:
rng = np.random.default_rng(1)
X, y = separable(200, rng, gap=0.0)
flip = rng.random(len(y)) < 0.05                    # немного шума в метках
y = np.where(flip, -y, y)
C = 1.0

def objective(w, b):
    return 0.5 * w @ w + C * np.maximum(0, 1 - y * (X @ w + b)).sum()

w, b = np.zeros(2), 0.0
for t in range(1, 20001):
    margin = y * (X @ w + b)
    viol = margin < 1                               # кто даёт ненулевой вклад в hinge
    grad_w = w - C * (y[viol, None] * X[viol]).sum(0)
    grad_b = -C * y[viol].sum()
    step = 0.5 / (t + 100)
    w -= step * grad_w
    b -= step * grad_b

sk = SVC(kernel="linear", C=C).fit(X, y)
w_sk, b_sk = sk.coef_[0], sk.intercept_[0]
agree = np.mean(np.sign(X @ w + b) == np.sign(X @ w_sk + b_sk))
print(f"целевая функция: субградиент {objective(w, b):.3f}, sklearn {objective(w_sk, b_sk):.3f}")
print(f"cos(w, w_sklearn) = {w @ w_sk / (np.linalg.norm(w) * np.linalg.norm(w_sk)):.5f}")
print(f"совпадение предсказаний на обучении: {agree:.3f}")
print(f"объектов с ненулевой hinge loss: {(y * (X @ w_sk + b_sk) < 1).sum()} из {len(y)}")

целевая функция: субградиент 48.515, sklearn 48.516
cos(w, w_sklearn) = 1.00000
совпадение предсказаний на обучении: 1.000
объектов с ненулевой hinge loss: 49 из 200


## 3. Ядерный трюк численно

Для полиномиального ядра степени 2 явное отображение известно: все мономы степени
не выше 2 с коэффициентами $\sqrt 2$. Проверим, что $K(x, z) = (x\cdot z + 1)^2$
совпадает с $\langle\phi(x), \phi(z)\rangle$, и сравним стоимость.

In [4]:
def phi_poly2(X):
    '''Явное отображение для (x·z + 1)^2.'''
    n, d = X.shape
    feats = [np.ones(n)]
    feats += [np.sqrt(2) * X[:, i] for i in range(d)]
    feats += [X[:, i] ** 2 for i in range(d)]
    feats += [np.sqrt(2) * X[:, i] * X[:, j] for i in range(d) for j in range(i + 1, d)]
    return np.column_stack(feats)

rng = np.random.default_rng(2)
for d in (3, 50, 300):
    A = rng.normal(size=(200, d)); B = rng.normal(size=(200, d))
    t = time.perf_counter(); K = (A @ B.T + 1) ** 2; t_kernel = time.perf_counter() - t
    t = time.perf_counter(); PA, PB = phi_poly2(A), phi_poly2(B); K_exp = PA @ PB.T
    t_explicit = time.perf_counter() - t
    print(f"d = {d:3d}: размер φ(x) = {PA.shape[1]:6d}, "
          f"max|K - φ·φ| = {np.abs(K - K_exp).max():.1e}, "
          f"ядро {t_kernel * 1e3:6.2f} мс, явно {t_explicit * 1e3:8.2f} мс")

d =   3: размер φ(x) =     10, max|K - φ·φ| = 5.7e-14, ядро   0.86 мс, явно     0.30 мс
d =  50: размер φ(x) =   1326, max|K - φ·φ| = 9.1e-13, ядро   0.28 мс, явно    24.59 мс


d = 300: размер φ(x) =  45451, max|K - φ·φ| = 7.7e-12, ядро   4.79 мс, явно   756.60 мс


## 4. Vanilla SVM на четырёх наборах и опорные векторы

In [5]:
rng = np.random.default_rng(11)
X1, y1 = separable(300, rng)
X3, y3 = make_circles(400, noise=0.08, factor=0.45, random_state=1)
X4, y4 = make_moons(400, noise=0.18, random_state=2)
sets = {"разделимые": (X1, y1), "окружности": (X3, y3), "полумесяцы": (X4, y4)}

print(f"{'набор':<12}{'линейный':>10}{'RBF':>8}{'опорных у RBF':>16}")
for name, (X, y) in sets.items():
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    lin = SVC(kernel="linear", C=1.0).fit(Xtr, ytr)
    rbf = SVC(kernel="rbf", C=1.0, gamma="scale").fit(Xtr, ytr)
    print(f"{name:<12}{lin.score(Xte, yte):>10.3f}{rbf.score(Xte, yte):>8.3f}"
          f"{rbf.n_support_.sum() / len(ytr):>15.0%}")

print("\nполумесяцы с шумом: как γ и C меняют число опорных векторов")
Xm, ym = make_moons(500, noise=0.25, random_state=3)
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.5, random_state=0, stratify=ym)
print(f"{'γ':>8}{'C':>8}{'train':>8}{'test':>8}{'опорных':>10}")
for g in (0.01, 1, 18, 1000):
    for C in (0.1, 10):
        m = SVC(kernel="rbf", gamma=g, C=C).fit(Xtr, ytr)
        print(f"{g:>8}{C:>8}{m.score(Xtr, ytr):>8.3f}{m.score(Xte, yte):>8.3f}"
              f"{m.n_support_.sum() / len(ytr):>9.0%}")

набор         линейный     RBF   опорных у RBF
разделимые       1.000   1.000            10%
окружности       0.550   1.000            14%
полумесяцы       0.844   0.963            20%

полумесяцы с шумом: как γ и C меняют число опорных векторов
       γ       C   train    test   опорных
    0.01     0.1   0.792   0.836     100%
    0.01      10   0.848   0.880      44%
       1     0.1   0.908   0.932      57%
       1      10   0.956   0.932      19%
      18     0.1   0.952   0.928      99%
      18      10   0.964   0.940      47%
    1000     0.1   1.000   0.680     100%
    1000      10   1.000   0.712     100%


## 5. Отступы AdaBoost: худшие растут, медиана падает — пока метки чистые

Один и тот же AdaBoost на двух версиях данных: без шума в метках и с 1% перевёрнутых
меток (это, кстати, значение `flip_y` по умолчанию в `make_classification`).

In [6]:
def ada_margins(flip_y):
    X, y = make_classification(n_samples=1500, n_features=10, n_informative=6,
                               n_redundant=2, class_sep=1.0, flip_y=flip_y,
                               random_state=3)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, random_state=0)
    ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=2), n_estimators=600,
                             learning_rate=0.5, random_state=0).fit(Xtr, ytr)
    a = ada.estimator_weights_[: len(ada.estimators_)]
    H_tr = np.array([2 * e.predict(Xtr) - 1 for e in ada.estimators_])
    H_te = np.array([2 * e.predict(Xte) - 1 for e in ada.estimators_])
    ys, yts = 2 * ytr - 1, 2 * yte - 1
    rows = []
    for T in (5, 50, 200, 400, 600):
        F_tr = (a[:T, None] * H_tr[:T]).sum(0) / a[:T].sum()
        F_te = (a[:T, None] * H_te[:T]).sum(0) / a[:T].sum()
        m = ys * F_tr
        rows.append((T, np.mean(m <= 0), np.mean(yts * F_te <= 0), m.min(), np.median(m)))
    return rows

for flip in (0.0, 0.01):
    print(f"доля перевёрнутых меток: {flip:.0%}")
    print(f"{'T':>5}{'ошибка обуч.':>14}{'ошибка тест':>13}{'мин. отступ':>13}{'медиана':>10}")
    for T, tr, te, mn, med in ada_margins(flip):
        print(f"{T:>5}{tr:>14.3f}{te:>13.3f}{mn:>+13.3f}{med:>+10.3f}")
    print()

доля перевёрнутых меток: 0%
    T  ошибка обуч.  ошибка тест  мин. отступ   медиана


    5         0.115        0.132       -0.686    +0.686
   50         0.053        0.089       -0.179    +0.384
  200         0.008        0.073       -0.020    +0.258
  400         0.000        0.068       +0.027    +0.224
  600         0.000        0.067       +0.049    +0.208

доля перевёрнутых меток: 1%
    T  ошибка обуч.  ошибка тест  мин. отступ   медиана


    5         0.109        0.145       -0.762    +0.712
   50         0.052        0.108       -0.196    +0.423
  200         0.019        0.089       -0.048    +0.294
  400         0.001        0.081       -0.002    +0.229
  600         0.000        0.091       +0.035    +0.201



Без шума тест монотонно улучшается и после нулевой ошибки на обучении — отступы
растут, как и обещает теория. С одним процентом перевёрнутых меток картина другая:
чтобы довести ошибку на обучении до нуля, бустингу приходится «выучить» неправильные
метки, и тестовая ошибка разворачивается вверх. Худший отступ в зашумлённых данных —
это ошибочно размеченный объект, и именно в него алгоритм вкладывается.

## 6. Бустинг не экстраполирует

Дерево кусочно-постоянно, поэтому за пределами обучающего диапазона сумма деревьев —
константа. Обучим на $y = 2x + \varepsilon$ для $x \in [0, 10]$ и спросим о $x = 15$ и $x = 30$.

In [7]:
rng = np.random.default_rng(4)
x = rng.uniform(0, 10, 500)
y_reg = 2 * x + rng.normal(0, 0.5, 500)

gbm = HistGradientBoostingRegressor(random_state=0).fit(x[:, None], y_reg)
lin = LinearRegression().fit(x[:, None], y_reg)

x_new = np.array([[5.0], [9.9], [15.0], [30.0], [100.0]])
print(f"{'x':>7}{'истина 2x':>11}{'бустинг':>10}{'линейная':>10}")
for xv, g, l in zip(x_new.ravel(), gbm.predict(x_new), lin.predict(x_new)):
    print(f"{xv:>7.1f}{2 * xv:>11.1f}{g:>10.2f}{l:>10.2f}")

      x  истина 2x   бустинг  линейная
    5.0       10.0     10.10      9.99
    9.9       19.8     19.47     19.79
   15.0       30.0     19.47     30.00
   30.0       60.0     19.47     60.01
  100.0      200.0     19.47    200.08


## 7. Что осталось в руках

- Жёсткий SVM — это обычная выпуклая квадратичная задача: оптимизатор общего назначения
  находит те же опорные векторы и ту же гиперплоскость, что и sklearn.
- Мягкий SVM — это hinge loss плюс $L_2$: субградиентный спуск без всякой двойственности
  приходит к тому же решению.
- Ядро $(x\cdot z + 1)^2$ совпадает со скалярным произведением явных мономов до машинной
  точности, а стоит на порядки дешевле уже при сотнях признаков.
- Большое $\gamma$ превращает все объекты в опорные векторы и запоминает выборку.
- AdaBoost после нулевой ошибки на обучении поднимает минимальный отступ и опускает
  медиану — оптимизирует худший случай. На чистых метках это улучшает тест, но уже 1%
  перевёрнутых меток разворачивает тестовую ошибку вверх.
- За пределами обучающего диапазона бустинг выдаёт константу; линейная модель продолжает
  тренд.